# Escalado de Features

Cuando las variables tienen escalas muy distintas, el gradiente descendente zigzaguea en lugar de converger directo al mínimo. En este notebook veremos por qué sucede eso, cómo visualizarlo, y cómo corregirlo.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Estilo oscuro coherente con el resto del curso
plt.rcParams.update({
    'figure.facecolor': '#0f0f11',
    'axes.facecolor':   '#0f0f11',
    'axes.edgecolor':   '#2a2a30',
    'axes.labelcolor':  '#8a8a9a',
    'xtick.color':      '#8a8a9a',
    'ytick.color':      '#8a8a9a',
    'text.color':       '#e2e2e8',
    'grid.color':       '#2a2a30',
    'grid.linestyle':   '--',
    'grid.alpha':       0.5,
})

# Carpeta donde se guardan las imágenes para el HTML
# El notebook vive en notebooks/, el HTML en docs/ → los assets van en docs/assets/
ASSETS_DIR = Path("../docs/assets/feature-scaling")
ASSETS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Imágenes se guardarán en: {ASSETS_DIR.resolve()}")

## 1. El problema de las escalas

Generamos dos features con rangos muy distintos: metros cuadrados (500–5000) y número de habitaciones (1–8).

In [ ]:
np.random.seed(42)
n = 200

metros       = np.random.uniform(500, 5000, n)
habitaciones = np.random.randint(1, 9, n).astype(float)

# Variable objetivo: precio de la casa
precio = 0.05 * metros + 20 * habitaciones + np.random.randn(n) * 80

print("Feature          min      max      media    std")
print("-" * 55)
for nombre, arr in [("metros", metros), ("habitaciones", habitaciones), ("precio", precio)]:
    print(f"{nombre:<16} {arr.min():6.1f}   {arr.max():6.1f}   {arr.mean():6.1f}   {arr.std():6.1f}")

## 2. Visualizar los contornos de la función de costo

Los contornos muestran qué forma tiene la superficie de pérdida.
- **Sin escalar**: elipses muy alargadas → el gradiente zigzaguea.
- **Con Z-score**: contornos casi circulares → el gradiente apunta directo al mínimo.

In [ ]:
def zscore(x):
    return (x - x.mean()) / x.std()

def ecm_grid(w1, w2, X1, X2, y):
    return ((y - (w1 * X1 + w2 * X2)) ** 2).mean()

metros_z       = zscore(metros)
habitaciones_z = zscore(habitaciones)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Contornos de la función de costo", fontsize=13)

configs = [
    (metros,   habitaciones,   np.linspace(0.03, 0.07, 300), np.linspace(10, 30, 300), "Sin escalar"),
    (metros_z, habitaciones_z, np.linspace(-5,   5,    300), np.linspace(-5,  5, 300), "Con Z-score"),
]

for ax, (x1, x2, wa_range, wb_range, titulo) in zip(axes, configs):
    WA, WB = np.meshgrid(wa_range, wb_range)
    Z = np.vectorize(lambda a, b: ecm_grid(a, b, x1, x2, precio))(WA, WB)
    ax.contourf(WA, WB, Z, levels=30, cmap="Blues", alpha=0.6)
    ax.contour(WA, WB, Z,  levels=15, colors="#6c7fea", linewidths=0.6, alpha=0.7)
    ax.set_title(titulo, fontsize=11)
    ax.set_xlabel("w₁  (metros)")
    ax.set_ylabel("w₂  (habitaciones)")

plt.tight_layout()
plt.savefig(ASSETS_DIR / "contornos.png", dpi=150, bbox_inches="tight", facecolor="#0f0f11")
print("Guardado: contornos.png")
plt.show()

## 3. Los tres métodos de escalado

Implementamos las tres fórmulas manualmente y comparamos visualmente su efecto.

In [ ]:
def escalar_minmax(x):
    """Resultado en [0, 1]."""
    return (x - x.min()) / (x.max() - x.min())

def normalizar_media(x):
    """Centrado en 0, resultado aproximadamente en [-0.5, 0.5]."""
    return (x - x.mean()) / (x.max() - x.min())

def zscore(x):
    """Media 0, desviación estándar 1."""
    return (x - x.mean()) / x.std()

versiones = [
    (metros,                   "Original"),
    (escalar_minmax(metros),   "Min-Max"),
    (normalizar_media(metros), "Media"),
    (zscore(metros),           "Z-score"),
]

fig, axes = plt.subplots(1, 4, figsize=(15, 4))
fig.suptitle("Efecto de cada método de escalado sobre metros cuadrados", fontsize=12)

for ax, (d, titulo) in zip(axes, versiones):
    ax.hist(d, bins=30, color="#6c7fea", alpha=0.8, edgecolor="#2a2a30", linewidth=0.4)
    ax.set_title(titulo, fontsize=10)
    ax.set_xlabel(f"μ={d.mean():.2f}  σ={d.std():.2f}", fontsize=8)

plt.tight_layout()
plt.savefig(ASSETS_DIR / "histogramas.png", dpi=150, bbox_inches="tight", facecolor="#0f0f11")
print("Guardado: histogramas.png")
plt.show()

print("Nota: la FORMA del histograma no cambia, solo los valores del eje x.")

## 4. Gradiente descendente: con y sin escalado

Entrenamos regresión lineal múltiple con gradiente descendente en ambas versiones.
Compara cuántas iteraciones necesita cada una y con qué learning rate.

In [ ]:
def gradiente_paso(w, alpha, X, y):
    """Un paso de gradiente descendente (batch) para regresión lineal."""
    n = len(y)
    pred = X @ w
    grad = (2 / n) * X.T @ (pred - y)
    return w - alpha * grad

def entrenar(X, y, alpha, n_iter=600):
    """Devuelve los pesos finales y el historial de ECM."""
    w = np.zeros(X.shape[1])
    hist = []
    for _ in range(n_iter):
        pred = X @ w
        hist.append(((y - pred) ** 2).mean())
        w = gradiente_paso(w, alpha, X, y)
    return w, hist

# Preparar matrices (con columna de bias)
bias  = np.ones(n)
X_raw = np.column_stack([bias, metros,          habitaciones])
X_sc  = np.column_stack([bias, zscore(metros),  zscore(habitaciones)])

w_raw, hist_raw = entrenar(X_raw, precio, alpha=1e-9)
w_sc,  hist_sc  = entrenar(X_sc,  precio, alpha=0.15)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("Convergencia del gradiente descendente", fontsize=12)

axes[0].plot(hist_raw, color="#ea6c6c", linewidth=1.5)
axes[0].set_title("Sin escalar  (α = 1e-9)", fontsize=10)
axes[0].set_xlabel("Iteración")
axes[0].set_ylabel("ECM")

axes[1].plot(hist_sc, color="#6c7fea", linewidth=1.5)
axes[1].set_title("Con Z-score  (α = 0.15)", fontsize=10)
axes[1].set_xlabel("Iteración")
axes[1].set_ylabel("ECM")

plt.tight_layout()
plt.savefig(ASSETS_DIR / "convergencia.png", dpi=150, bbox_inches="tight", facecolor="#0f0f11")
print("Guardado: convergencia.png")
plt.show()

print(f"ECM final sin escalar: {hist_raw[-1]:.2f}")
print(f"ECM final con Z-score: {hist_sc[-1]:.2f}")

## 5. Cómo escalar correctamente: train/test

**Regla**: ajustar el escalador solo con datos de entrenamiento, luego aplicarlo a ambos conjuntos.
Si usas el dataset completo para calcular la media y std, filtras información del test al entrenamiento (data leakage).

In [ ]:
# Construir matriz de features
X = np.column_stack([metros, habitaciones])

# 1. Dividir PRIMERO
X_train, X_test, y_train, y_test = train_test_split(
    X, precio, test_size=0.2, random_state=42
)

# 2. Ajustar escalador SOLO sobre train
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)

# 3. Aplicar a test usando los parámetros de train
X_test_sc = scaler.transform(X_test)

print("Parámetros aprendidos del entrenamiento:")
print(f"  Media:  {scaler.mean_}")
print(f"  Std:    {scaler.scale_}")
print()
print("Train escalado — media por feature:", X_train_sc.mean(axis=0).round(4))
print("Test  escalado — media por feature:", X_test_sc.mean(axis=0).round(4))
print()
print("(El test no tiene media exactamente 0 — eso es correcto y esperado)")

## 6. Ejercicios

1. Carga el dataset `data/csvs/examenes.csv`. Elige dos variables numéricas y visualiza sus distribuciones. ¿Tienen rangos similares?

2. Aplica los tres métodos de escalado (min-max, media, z-score) a las variables que elegiste. Grafica los histogramas comparados.

3. Entrena un modelo de regresión lineal múltiple con gradiente descendente usando esas dos variables como features. Compara la convergencia con y sin escalado. ¿Cuál learning rate usas en cada caso?

4. Implementa el flujo completo: divide en train/test, ajusta el escalador solo en train y aplícalo a ambos conjuntos. Evalúa el ECM en test.

In [ ]:
# Ejercicio 1
df = pd.read_csv("../data/csvs/examenes.csv")
df.head()

In [ ]:
# Ejercicio 2 — tu código aquí


In [ ]:
# Ejercicio 3 — tu código aquí


In [ ]:
# Ejercicio 4 — tu código aquí
